In [ ]:
# import libraries
import numpy as np
import scipy as sp
import pickle
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

In [ ]:
# Categorization functions

# def categorize_diffusers(coordinates):
#     """Recieves a coordinates array of shape [n_diffusers, 3], and returns an array of shape [n_diffusers],
#     Where each entry is in the range [0,9] (inclusive). Which signify the following state states:
#     0: Cytoplasm
#     1-8: NPC radial segmets
#     9: Nucleus
#     """
#     n_diffusers = coordinates.shape[0]
#     categorized_coordinates = np.zeros(shape=(n_diffusers))
    
#     # Create masks
#     cyto_mask = coordinates[:, 2] >= 15
#     nuc_mask = coordinates[:, 2] <= -15
    
#     # apply masks
#     angle = np.arctan2(coordinates[:, 1], coordinates[:, 0]) + np.pi
#     eighth = np.floor(angle / (np.pi * 0.25)) + 1
#     eighth[eighth == 9] = 8
#     categorized_coordinates[((~cyto_mask) & (~nuc_mask))] = eighth[((~cyto_mask) & (~nuc_mask))]
#     categorized_coordinates[cyto_mask] = 0
#     categorized_coordinates[nuc_mask] = 9
    
#     return categorized_coordinates


# def categorize_diffusers2(coordinates, angle_offset=0):
#     """
#     Recieves a coordinates array of shape [n_diffusers, 3], and returns an array of shape [n_diffusers],
#     Where each entry is in the range [0,17] (inclusive). Which signify the following state states:
#     0: Cytoplasm
#     1-8: NPC radial segmets (cytoplasm half)
#     9-16: NPC radial segmets (nucleus half)
#     17: Nucleus
#     angle_offset should be a number between 0 and 0.25*pi, signifying the angle offset for spoke barriers.
#     """
#     n_diffusers = coordinates.shape[0]
#     categorized_coordinates = np.zeros(shape=(n_diffusers))
    
#     # Create masks
#     cyto_mask = coordinates[:, 2] >= 15
#     nuc_mask = coordinates[:, 2] <= -15
#     cyto_half_mask = coordinates[:, 2] >= 0
    
#     # apply masks
#     angle = np.arctan2(coordinates[:, 1], coordinates[:, 0]) + np.pi + angle_offset
#     angle = np.mod(angle, 2 * np.pi)
#     eighth = np.floor(angle / (np.pi * 0.25)) + 1
#     eighth[eighth == 9] = 8
#     categorized_coordinates[((~cyto_mask) & (~nuc_mask) & cyto_half_mask)] = eighth[((~cyto_mask) & (~nuc_mask) & cyto_half_mask)]
#     categorized_coordinates[((~cyto_mask) & (~nuc_mask) & (~cyto_half_mask))] = eighth[((~cyto_mask) & (~nuc_mask) & (~cyto_half_mask))] + 8
#     categorized_coordinates[cyto_mask] = 0
#     categorized_coordinates[nuc_mask] = 17
    
#     return categorized_coordinates

# def categorize_diffusers3(coordinates, angle_offset=0):
#     """
#     Recieves a coordinates array of shape [n_diffusers, 3], and returns an array of shape [n_diffusers],
#     Where each entry is in the range [0,25] (inclusive). Which signify the following state states:
#     0: Cytoplasm
#     1-8: NPC radial segmets (cytoplasm third)
#     9-16: NPC radial segments (center third)
#     17-24: NPC radial segmets (nucleus third)
#     25: Nucleus
#     angle_offset should be a number between 0 and 0.25*pi, signifying the angle offset for spoke barriers.
#     """
#     n_diffusers = coordinates.shape[0]
#     categorized_coordinates = np.zeros(shape=(n_diffusers))
    
#     # Create masks
#     cyto_mask = coordinates[:, 2] >= 15
#     nuc_mask = coordinates[:, 2] <= -15
#     cyto_third_mask = coordinates[:, 2] >= 5
#     nuc_third_mask = coordinates[:, 2] <= -5
#     center_third_mask = ~cyto_third_mask & ~nuc_third_mask
    
    
#     # apply masks
#     angle = np.arctan2(coordinates[:, 1], coordinates[:, 0]) + np.pi + angle_offset
#     angle = np.mod(angle, 2 * np.pi)
#     eighth = np.floor(angle / (np.pi * 0.25)) + 1
#     eighth[eighth == 9] = 8
#     categorized_coordinates[((~cyto_mask) & (~nuc_mask) & cyto_third_mask)] = eighth[((~cyto_mask) & (~nuc_mask) & cyto_third_mask)]
#     categorized_coordinates[((~cyto_mask) & (~nuc_mask) & center_third_mask)] = eighth[((~cyto_mask) & (~nuc_mask) & center_third_mask)] + 8
#     categorized_coordinates[((~cyto_mask) & (~nuc_mask) & nuc_third_mask)] = eighth[((~cyto_mask) & (~nuc_mask) & nuc_third_mask)] + 16
#     categorized_coordinates[cyto_mask] = 0
#     categorized_coordinates[nuc_mask] = 25
    
#     return categorized_coordinates

def categorize_diffusers_n(coordinates, n, angle_offset=0):
    """
    Recieves a coordinates array of shape [n_diffusers, 3], and returns an array of shape [n_diffusers],
    Where each entry is in the range [0,n*8 + 1] (inclusive). Which signify the following state states:
    0: Cytoplasm
    i+1-i+9: 8 NPC radial segmets in the i'th vertical layer
    n*8 + 1: Nucleus
    angle_offset should be a number between 0 and 0.25*pi, signifying the angle offset for spoke barriers.
    """
    n_diffusers = coordinates.shape[0]
    categorized_coordinates = np.zeros(shape=(n_diffusers))
    
    # Create masks
    cyto_mask = coordinates[:, 2] >= 15
    nuc_mask = coordinates[:, 2] <= -15
    
    # Calculate the vertical layer boundaries
    layer_size = 30 / n  # 30 is the total height (-15 to 15)
    layer_boundaries = np.linspace(-15, 15, n+1)
    
    # Calculate angles for radial segmentation
    angle = np.arctan2(coordinates[:, 1], coordinates[:, 0]) + np.pi + angle_offset
    angle = np.mod(angle, 2 * np.pi)
    eighth = np.floor(angle / (np.pi * 0.25)) + 1
    eighth[eighth == 9] = 8  # Handle edge case
    
    # Apply cytoplasm and nucleus masks first
    categorized_coordinates[cyto_mask] = 0
    categorized_coordinates[nuc_mask] = n*8 + 1
    
    # For each vertical layer, categorize the diffusers
    for i in range(n):
        layer_mask = (~cyto_mask & ~nuc_mask & 
                      (coordinates[:, 2] >= layer_boundaries[i]) & 
                      (coordinates[:, 2] < layer_boundaries[i+1]))
        
        # Assign values for this layer: i*8 + eighth (with offset adjustment)
        categorized_coordinates[layer_mask] = i*8 + eighth[layer_mask]
    
    return categorized_coordinates

    
def categorize_diffusers_over_time(trajectories, angle_offset, step=1, n_layers=1):
    n_diffusers = trajectories.shape[0]
    n_t = trajectories.shape[2]
    n_columns = len(range(0, n_t, step))
    categorized_trajectories = np.zeros(shape=(n_diffusers, n_columns))
    for t in range(0, n_t, step):
        categorized_trajectories[:,int(t / step)] = categorize_diffusers_n(trajectories[:, :, t], n_layers, angle_offset=angle_offset)
    return categorized_trajectories, n_layers * 8 + 2

In [ ]:
# Transition matrix functions

from numpy import float64


def normalize_rows(counts_matrix):
    transition_matrix = np.zeros_like(counts_matrix)
    n_states = counts_matrix.shape[0]
    for i in range(n_states):
        row_sum = np.sum(counts_matrix[i, :])
        if row_sum == 0:
            print(f"Row sum is 0, setting row {i} to 0    :( ")
            transition_matrix[i, :] = 0
            continue
        transition_matrix[i, :] = counts_matrix[i, :] / row_sum
    return transition_matrix
    

def calc_counts_matrix(categorized_trajectories, n_states, init_1 = False):
    if init_1: counts_matrix = np.ones(shape=(n_states,n_states))
    else: counts_matrix = np.zeros(shape=(n_states,n_states))
    n_diffusers = categorized_trajectories.shape[0]
    n_t = categorized_trajectories.shape[1]
    
    for i in range(n_diffusers):
        for t in range(n_t - 1):
            first = int(categorized_trajectories[i,t])
            second = int(categorized_trajectories[i,t+1])
            counts_matrix[first, second] += 1
    return counts_matrix

def eightwise_symmetrize(data):
    """Given a matrix, makes it 8-wise symmetric"""
    if (data.shape[0] % 8 != 0) or (data.shape[1] % 8 != 0):
        raise Exception("invalid matrix shape")
    mask_base = np.array(np.eye(8, dtype=bool))
    masks = [np.roll(mask_base, shift=i, axis=0) for i in range(8)]
    new_data = np.zeros_like(data)
    for mask in masks:
        for i in range(int(data.shape[0] / 8)):
            for j in range(int(data.shape[1] / 8)):
                new_data[i*8:(i+1)*8, j*8:(j+1)*8][mask] += np.sum(data[i*8:(i+1)*8, j*8:(j+1)*8][mask])
    return new_data

def calc_transition_matrix(categorized_trajectories, n_states, symmetrize=False, init_1 = False):
    counts_matrix = calc_counts_matrix(categorized_trajectories, n_states, init_1=init_1)
    if symmetrize: counts_matrix[1:-1, 1:-1] = eightwise_symmetrize(counts_matrix[1:-1, 1:-1])
    transition_matrix = normalize_rows(counts_matrix)
    return transition_matrix

# def calc_radialized_transition_matrix(categorized_trajectories, categorization_version, init_1 = False):
#     """ 
#     Given categorized trajectories of version 3, computes a transition matrix where the transition for each spoke is the same,
#     shifted radially, through averaging.
#     """
#     if categorization_version != 3:
#         raise Exception("Invalid Version")
#     counts_matrix = calc_counts_matrix(categorized_trajectories, 26, init_1)
#     avgd_coords = []
#     orig = np.array(range(8))
#     for i in range(8):
#         avgd_coords.append((orig + i) % 8 + 1)
#         avgd_coords.append((orig + i) % 8 + 9)
#         avgd_coords.append((orig + i) % 8 + 17)
#     for i in range(len(avgd_coords)):
#         counts_matrix[avgd_coords[i], avgd_coords[i]] = np.mean(counts_matrix[avgd_coords[i], avgd_coords[i]])
#     for to_avg_range in [list(range(1, 9)), list(range(9, 17)), list(range(17, 25))]:
#         counts_matrix[to_avg_range, 0] = np.mean(counts_matrix[to_avg_range, 0])
#         counts_matrix[to_avg_range, 25] = np.mean(counts_matrix[to_avg_range, 25])
#         counts_matrix[0, to_avg_range] = np.mean(counts_matrix[0, to_avg_range])
#         counts_matrix[25, to_avg_range] = np.mean(counts_matrix[25, to_avg_range])
#     transition_matrix = normalize_rows(counts_matrix)
#     return transition_matrix
    

def infinitesimal_generator(P, dt=1.0):
    """
    Estimate the infinitesimal generator matrix Q from transition matrix P
    Implementation similar to: https://transitionmatrix.readthedocs.io/en/latest/_modules/transitionMatrix/model.html#TransitionMatrix.generator
    Units of Q are rate, i.e. 1/time. 
    expm(Q * dt) will result in P.
    
    Examples:
    If the units of P are transition probablities per 100ns,
    then the units of Q will be the transition rate 1 / (100ns * dt).
    
    
    Parameters:
    P : numpy.ndarray
        Transition matrix
    dt : float, optional
        The time scale parameter. 
    
    Returns:
    numpy.ndarray
        Infinitesimal generator matrix Q. 
    """
    return sp.linalg.logm(P) / dt
    
def transition_from_generator(Q, dt=1.0):
    return sp.linalg.expm(Q * dt)

def visualize_transition_matrix(data, ticks_list, title, vlines=None, hlines=None,
                                cbar_label=r'$Log_{10} (Pr[i \to j])$', suptitle="Spatial Transition Matrix",
                                cmap="jet", diverge_colors_around_0=False, first_and_last_factor=1):
        
    # Expand first and last rows
    first_row_repeated = np.tile(data[0, :], (first_and_last_factor-1, 1))
    data = np.append(first_row_repeated, data, axis=0)
    first_col_repeated = np.tile(data[:, 0], (first_and_last_factor-1, 1))
    data = np.append(first_col_repeated.T, data, axis=1)
    last_row_repeated = np.tile(data[-1, :], (first_and_last_factor-1, 1))
    data = np.append(data, last_row_repeated, axis=0)
    last_col_repeated = np.tile(data[:, -1], (first_and_last_factor-1, 1))
    data = np.append(data, last_col_repeated.T, axis=1)
    
    if diverge_colors_around_0:
        plt.imshow(data, cmap=cmap, interpolation='nearest', norm=TwoSlopeNorm(0))
    else:
        plt.imshow(data, cmap=cmap, interpolation='nearest')
    
    # Set ticks
    tick_positions = np.array(range(len(ticks_list)), dtype=float64)
    first_pos = (first_and_last_factor - 1) / 2
    tick_positions += first_and_last_factor - 1
    tick_positions[0] = first_pos
    tick_positions[-1] += first_pos
    plt.xticks(tick_positions, ticks_list, rotation="vertical")
    plt.yticks(tick_positions, ticks_list)
    
    # add v and h lines
    if vlines is not None: plt.vlines(np.array(vlines) + first_and_last_factor - 1, ymin=-0.5, ymax=data.shape[0]-0.5, colors="black")
    if hlines is not None: plt.hlines(np.array(hlines) + first_and_last_factor - 1, xmin=-0.5, xmax=data.shape[1]-0.5, colors="black")
    # set labels and titles
    plt.xlabel("To") # checked this
    plt.ylabel("From")
    plt.suptitle(suptitle)
    plt.title(title)
    # add colorbar
    cbar = plt.colorbar()
    cbar.set_label(cbar_label)
    plt.show()


In [ ]:
# Process data
categorized_trajectories, n_states = categorize_diffusers_over_time(trajectories=trajectories,
                                                                    angle_offset=3,
                                                                    n_layers=10)
tm = calc_transition_matrix(categorized_trajectories=categorized_trajectories,
                            n_states=n_states,
                            symmetrize=True,
                            init_1=True)
with open("data/transition_matrices/spatial-150-180-symmetric-10layers.pickle", "wb") as f:
    pickle.dump(tm, f)